Overview:

Exports:
Student/Year Data out of Primero Edge
Student Contact Data our of PowerSchool SIS

Preperation:
Clean Student data from Primero Edge
-Remove Pre-Registrations for Next Year

Clean Contact data from Powerschool SIS
-Identify preferred contact types
-Prep/Clean data so only those contact types exist

Manipulate:
Merge together data sets so that for each student in the Primero Edge Export, there is only 1 primary contact from the Student Contacts list out of SIS

Authenticate:
Review there are no blank cells for each required field
Validate all students are accounted for compared to active list of students enrolled

In [7]:
# -----Libraries, Files and Paths-----

# LIBRARIES
import pandas as pd
import re  # Import regular expressions library
import os # Import OS library (Miscellaneous operating system interfaces)
from warnings import filterwarnings

#FILES --- UPDATE AS NEEDED
Primero_Edge_Data = "Summer EBT Enrollment Data.xlsx" # Update File name if needed to match your Primero Edge Export
Student_Enrollment_Data = "Student Enrollment Data.csv" # Update File name if needed to match your SIS Student Enrollment Export
Student_Contact_Data = "Student Contacts Information List.csv" # Update File name if needed to match your SIS Student Contact Information Export

# Automatic Directory Retrieval
# Get the user's current working directory
dir = os.getcwd()

# Suppress the excel/openpyxl warning
filterwarnings('ignore', category=UserWarning, module='openpyxl')

# Checks for Files in Directory
if os.path.isfile(os.path.join(dir, Primero_Edge_Data)): # Checks for Primero Edge File
  PE_Path = os.path.join(dir, Primero_Edge_Data) # If it exists, create the path
  print(f"PE_Path set to: {PE_Path}")
  PEDF = pd.read_excel(PE_Path) # import Primero Edge Student data
  print(f"{Primero_Edge_Data} succesfully uploaded.")
else:
  # If it doesn't exist, print a message and exit with an error code (1)
  print(f"File {Primero_Edge_Data} not found in {dir}")
  sys.exit(1)

if os.path.isfile(os.path.join(dir, Student_Enrollment_Data)): # Checks for SIS Student Enrollment File
  SE_Path = os.path.join(dir, Student_Enrollment_Data) # If it exists, create the path
  print(f"SE_Path set to: {SE_Path}")
  SEDF = pd.read_csv(SE_Path) # import SIS Student Enrollment data
  print(f"{Student_Enrollment_Data} succesfully uploaded.")
else:
  # If it doesn't exist, print a message and exit with an error code (1)
  print(f"File {Student_Enrollment_Data} not found in {dir}")
  sys.exit(1)

if os.path.isfile(os.path.join(dir, Student_Contact_Data)): # Checks for SIS Student Contact Information File
  # If it exists, create the path
  SC_Path = os.path.join(dir, Student_Contact_Data)
  print(f"SC_Path set to: {SC_Path}")
  SCDF = pd.read_csv(SC_Path) # import SIS Student Enrollment data
  print(f"{Student_Contact_Data} succesfully uploaded.")
else:
  # If it doesn't exist, print a message and exit with an error code (1)
  print(f"File {Student_Contact_Data} not found in {dir}")
  sys.exit(1)

# Set Path for Error Report
Error_Path = os.path.join(dir,"Errors.csv")

# Set Path for Final Report
Final_Path = os.path.join(dir,"Sunbuck Report.csv")

PE_Path set to: C:\Users\rkroker\Documents\Code\Python\Sunbuck\Summer EBT Enrollment Data.xlsx
Summer EBT Enrollment Data.xlsx succesfully uploaded.
SE_Path set to: C:\Users\rkroker\Documents\Code\Python\Sunbuck\Student Enrollment Data.csv
Student Enrollment Data.csv succesfully uploaded.
SC_Path set to: C:\Users\rkroker\Documents\Code\Python\Sunbuck\Student Contacts Information List.csv
Student Contacts Information List.csv succesfully uploaded.


Preperation Phase:

The files have now been uploaded and set to data frames to work within Python. The next steps are to start preparing the data frames to be merged into a final report. 

The Summer EBT Enrollment Data.xlsx has been uploaded and set to a data frame labled PE_Data. This Data Frame will be manipulated to remove the students who are pre-registered for next school year. This is done by using the students Date of Birth and their eligibility for KG in the State of Pennsylvania (4 Years, 7 Months is the absolute minimum age) against the start of the school year. This filtered data set will be labled Clean_PE_Data.

In [17]:
PEDF.head()

,*SFANameorLEAName,*AUN,*SchoolBuildingName,*School/Branch,*PASecureID,*StudentFirstName,StudentMiddleName,*StudentLastName,*StudentDateOfBirth,*MailingAddressLine1,...,*Eligibility,*AddressCountyName,Case Number (if available),*PrimaryGuardianFirstName,*PrimaryGuardianLastName,*PrimaryGuardianPhoneNumber,*PrimaryGuardianEmail,*EnrollmentBeginDate,*EnrollmentEndDate,ValidationOutput
0,PENNSBURY SCHOOL DISTRICT,122-09-820-2,MAKEFIELD EL SCH,1109,2300966,Josiah,NaN,Arzayus,2012-11-09,715 N Lafayette Ave,...,Free,Bucks,90400046.0,NaN,NaN,NaN,...,08/07/2023,NaN,
1,PENNSBURY SCHOOL DISTRICT,122-09-820-2,MAKEFIELD EL SCH,1109,2301046,Sarayah,NaN,Bowser,2016-02-29,250 Plaza Blvd Apt F6,...,Free,Bucks,NaN,NaN,NaN,NaN,lauren.clark36@yahoo.com ...,08/07/2023,NaN,
2,PENNSBURY SCHOOL DISTRICT,122-09-820-2,MAKEFIELD EL SCH,1109,2301059,Zamere,NaN,Gurganus,2016-08-27,250 Plaza BLVD A7,...,Paid,Bucks,NaN,NaN,NaN,NaN,talayamcmullen@gmail.com ...,08/07/2023,08/07/2023,
3,PENNSBURY SCHOOL DISTRICT,122-09-820-2,MAKEFIELD EL SCH,1109,2301118,Londyn,NaN,Hyman,2014-01-05,254 Trenton AVE 324A,...,Free,Bucks,90663036.0,NaN,NaN,NaN,...,08/07/2023,NaN,
4,PENNSBURY SCHOOL DISTRICT,122-09-820-2,MAKEFIELD EL SCH,1109,2301126,Shiloh,NaN,Angel,2014-02-26,567 Nancy RD,...,Paid,Bucks,NaN,NaN,NaN,NaN,...,08/07/2023,NaN,


In [21]:
#Clean Student data from Primero Edge
#-Remove Pre-Registrations for Next Year

# Minimum age requirement (4 years and 7 months)
min_age_in_years = 4
min_age_in_months = 7

# Start of school date (September 5th, 2023)
start_of_school = pd.to_datetime('2023-09-05')

# Define the ineligible birthdate threshold
ineligible_birthdate = pd.to_datetime("2018-08-30")

# Filter PEDF based on birthdate
Filtered_PEDF = PEDF[PEDF["*StudentDateOfBirth"].dt.date <= ineligible_birthdate.date()]

# Sort Clean_PE_Data by StudentDateOfBirth (descending = oldest first)
Clean_PEDF = Filtered_PEDF.sort_values(by='*StudentDateOfBirth', ascending=False)

# Further processing (optional)
# ... (your code using the filtered_PEDF DataFrame)
Clean_PEDF.head()

,*SFANameorLEAName,*AUN,*SchoolBuildingName,*School/Branch,*PASecureID,*StudentFirstName,StudentMiddleName,*StudentLastName,*StudentDateOfBirth,*MailingAddressLine1,...,*Eligibility,*AddressCountyName,Case Number (if available),*PrimaryGuardianFirstName,*PrimaryGuardianLastName,*PrimaryGuardianPhoneNumber,*PrimaryGuardianEmail,*EnrollmentBeginDate,*EnrollmentEndDate,ValidationOutput
1839,PENNSBURY SCHOOL DISTRICT,122-09-820-2,MANOR EL SCH,1107,2400124,Olivia,NaN,Hamilton,2018-08-30,2 Venture LN,...,Paid,Bucks,NaN,NaN,NaN,NaN,...,08/07/2023,NaN,
2464,PENNSBURY SCHOOL DISTRICT,122-09-820-2,AFTON EL SCH,7519,2400640,Wyatt,NaN,Seader,2018-08-29,9 Hillside LN,...,Paid,Bucks,NaN,NaN,NaN,NaN,...,08/07/2023,NaN,
3578,PENNSBURY SCHOOL DISTRICT,122-09-820-2,OXFORD VALLEY EL SCH,1106,2400396,Kori,NaN,Wiedeman,2018-08-28,674 Trenton RD,...,Paid,Bucks,NaN,NaN,NaN,NaN,...,08/07/2023,NaN,
1854,PENNSBURY SCHOOL DISTRICT,122-09-820-2,MANOR EL SCH,1107,2400322,Grayson,NaN,Braun,2018-08-27,191 Birch DR,...,Paid,Bucks,NaN,NaN,NaN,NaN,...,08/07/2023,NaN,
5254,PENNSBURY SCHOOL DISTRICT,122-09-820-2,PENN VALLEY EL SCH,1101,2400234,Owen,NaN,Flaherty,2018-08-27,33 East Ln,...,Paid,Bucks,NaN,NaN,NaN,NaN,...,08/07/2023,NaN,


The Student Contact Data.csv has been uploaded and set to a data frame labeled SC_Data. This Data Frame will need all inactive contacts removed. Then it will need to be manipulated to split the Name stored as FirstLast into multiple columns, First, Middle and Last. Lastly, any contact with missing required contact data will be sorted between two data frames, Clean_SC_Data and Invalid_SC_Data. The invalid data must be corrected in the system.

In [23]:
SCDF.head()

,School,Std Num,Student Name,Grade Level,Homeroom,Contact ID,Contact Name,Priority Order,Relationship,Original Contact Type,...,Contact Phone Type 2,Contact Phone Number 3,Contact Phone Type 3,Email Addresses,Has Data Access,Has Custody,Lives With,Receives Mail,Emerg Contact,School Pickup
0,PHS,1500256,"Aaron, Casey C",9,Room 228 - 8B,172901,Shara Aaron,1,Parent 1,NaN,...,Home,908-864-0523,Mobile,AARONSHARA@HOTMAIL.COM*,Yes,Yes,Yes,Yes,No,No
1,PHS,1500256,"Aaron, Casey C",9,Room 228 - 8B,172900,Harris Aaron,2,Parent 2,NaN,...,Mobile,-,Not Set,HARRISJAARON@HOTMAIL.COM*,Yes,No,No,Yes,No,No
2,PHS,1200220,"Aaron, Oliver B",12,NaN,172900,Harris Aaron,1,Parent 2,NaN,...,Mobile,-,Not Set,HARRISJAARON@HOTMAIL.COM*,Yes,Yes,Yes,Yes,No,No
3,PHS,1200220,"Aaron, Oliver B",12,NaN,172901,Shara Aaron,2,Parent 1,NaN,...,Home,908-864-0523,Mobile,AARONSHARA@HOTMAIL.COM*,Yes,No,No,Yes,No,No
4,PHS,1300063,"Abdellatif, Hibah I",11,NaN,169259,Aicha Bifoulloussane,1,Parent 1,NaN,...,Not Set,215-428-1237,Not Set,AICHABIFOU@HOTMAIL.COM*,Yes,No,No,Yes,No,No


In [117]:
#NamesDF = SCDF[['Contact ID', 'Contact Name', 'Contact Phone Number 1', 'Email Addresses']]
#NamesDF.head()

# Define a list of symbols
invalid_symbols = ["+", "-", "*", "/", "%", "^", "&", "=", "$", "!", "@", "#", "$", "%", "^", "&", "*", "(", ")", "_", "+", "=", "{", "}", "[", "]", "|", "\\", ";", ":", "'", ".", "<", ">", "?"]

# Define a function to check for invalid symbols
def has_invalid_symbols(name):
  return bool(re.search(r"[" + re.escape("".join(invalid_symbols)) + "]", name))

# Define a filtering condition for both columns having data
def has_both_data(row):
  return not (pd.isna(row['Contact Phone Number 1']) or pd.isna(row['Email Addresses']))

#Remove inactive contacts
Active_SCDF = SCDF[SCDF['Status'] != 'Inactive']

# Filter Active_SCDF using vectorized approach to remove invalid names
NamesDF = Active_SCDF[~Active_SCDF['Contact Name'].apply(has_invalid_symbols)]

# Filter SCDF and assign to NamesDF (if both columns have data)
NamesDF_valid = NamesDF[NamesDF.apply(has_both_data, axis=1)][['Contact ID', 'Contact Name','Priority Order', 'Contact Phone Number 1', 'Email Addresses']]

# Drop Duplicate contact IDs
NamesDF_dedupID = NamesDF_valid.drop_duplicates(subset="Contact ID", keep="first") 

# Drop Duplicate contact Names
NamesDF_dedupName = NamesDF_dedupID.drop_duplicates(subset="Contact Name", keep="first") 

# Get the number of rows in NamesDF
num_rows_in_SCDF = len(SCDF)
num_rows_in_Active_SCDF = len(Active_SCDF)
num_rows_in_NamesDF = len(NamesDF)
num_rows_in_NamesDF_valid = len(NamesDF_valid)
num_rows_in_NamesDF_dedupID = len(NamesDF_dedupID)
num_rows_in_NamesDF_dedupName = len(NamesDF_dedupName)

print(f"Number Contacts in SCDF: {num_rows_in_SCDF}")
print(f"Number Active Contacts in SCDF: {num_rows_in_Active_SCDF}")
print(f"Number Active Contacts without Invalid Names: {num_rows_in_namesdf}")
print(f"Number Active Valid Contacts after Phone Number and Email Validation: {num_rows_in_NamesDF_valid}")
print(f"Number Active valid Contacts after Depulication of Contact ID's: {num_rows_in_NamesDF_dedupID}")
print(f"Number Active valid Contacts after Depulication of Remaining Contact Names: {num_rows_in_NamesDF_dedupName}")

# Split names by spaces and expand into separate columns (up to 20)
NamesDF_Split = NamesDF_dedupID['Contact Name'].str.split(expand=True)

# Optional: Rename columns (modify max number of columns as needed)
NamesDF_Split.columns = [f"Name Part {i + 1}" for i in range(NamesDF_Split.shape[1])]

# Select desired columns from NamesDF_Split
selected_parts = NamesDF_Split[[f"Name Part {i}" for i in range(1, 5)]]  # Select parts 1 to 4

# Merge DataFrames based on Contact ID
merged_df = NamesDF_dedupName.merge(selected_parts, how='left', on='Contact ID')

# Print the first few rows (head)
merged_df.head()





Number Contacts in SCDF: 7820
Number Active Contacts in SCDF: 7812
Number Active Contacts without Invalid Names: 5139
Number Active Valid Contacts after Phone Number and Email Validation: 5016
Number Active valid Contacts after Depulication of Contact ID's: 4305
Number Active valid Contacts after Depulication of Remaining Contact Names: 4237


KeyError: 'Contact ID'

In [13]:
#Clean Contact data from Powerschool SIS
#Remove inactive contacts
Clean_SC_Data = SC_Data[SC_Data['Status'] != 'Inactive']

In [27]:
# Count the number of elements after splitting by whitespace
num_elements = SC_Data['Contact Name'].str.split().str.len()

# Print the value counts for the number of elements
print(num_elements.value_counts())

print(num_elements.head())


Contact Name
2    7584
3     221
4      15
Name: count, dtype: int64
0    2
1    2
2    2
3    2
4    2
Name: Contact Name, dtype: int64


In [45]:
def split_into_columns(name):
  """Splits a name string into separate columns based on the number of elements.

  Handles names with varying numbers of words (2 or 3).
  """
  split_parts = name.split()
  num_parts = len(split_parts)
  if num_parts > 3:  # Filter out names with more than 3 parts
    return None
  return pd.Series({f'Split{i+1}': split_parts[i] if i < num_parts else '' for i in range(4)})

# Filter SC_Data based on split name length
filtered_data = SC_Data[SC_Data['Contact Name'].apply(lambda x: len(x.split()) <= 3)]

# Create a new DataFrame with separate columns for valid names
Names = filtered_data['Contact Name'].apply(split_into_columns).reset_index(drop=True).drop('Split4', axis=1)

# Output to a CSV file named "split_names_valid.csv"
Names.to_csv("split_names_valid.csv", index=False)


In [39]:
def split_into_columns(name):
  """Splits a name string into separate columns based on the number of elements.

  Handles names with varying numbers of words (2, 3, or 4).
  """
  split_parts = name.split()
  num_parts = len(split_parts)
  return pd.Series({f'Split{i+1}': split_parts[i] if i < num_parts else '' for i in range(4)})

# Create a new DataFrame with separate columns
Names = Clean_SC_Data['Contact Name'].apply(split_into_columns).reset_index(drop=True)

# Output to a CSV file named "split_names.csv"
Names.to_csv("split_names.csv", index=False)






In [ ]:
def split_name(name):
  """Splits a name string into first, middle (empty if no middle name), and last name.

  Handles common formats, middle names, and potentially titles or suffixes.
  """
  if not name:
    return ['', '', '']  # Return empty strings for all parts if name is empty

  # Split on whitespace
  name_parts = name.split()

  # Handle common titles (consider expanding if needed)
  if name_parts[0] in ['Mr.', 'Dr.', 'Ms.']:
    name_parts = name_parts[1:]  # Remove title from name parts

  # Handle common suffixes (consider expanding if needed)
  if name_parts[-1] in ['Jr.', 'Sr.', 'III', 'II']:
    name_parts[-2] = name_parts[-2] + ' ' + name_parts[-1]  # Combine last name with suffix
    name_parts.pop()  # Remove suffix from list

  # Handle middle names (assuming max one middle name)
  if len(name_parts) == 3:
    return name_parts

  # Handle two-word names (assuming first and last)
  elif len(name_parts) == 2:
    return name_parts

  # Handle more complex names (potential for further logic using regular expressions)
  else:
    # Add logic for handling titles, multiple middle names, etc. (if applicable)
    # You might use regular expressions here to define patterns for splitting
    return ['', '', '']  # Placeholder for unhandled complex names

# Apply the split_name function with list comprehension
Clean_SC_Data[['First Name', 'Middle Name', 'Last Name']] = SC_Data['Contact Name'].apply(split_name)


In [ ]:
#Sort contact data with missing required contact data between two data frames, Clean_SC_Data and Invalid_SC_Data.

# Assuming the DataFrame has columns 'Contact Phone Number 1' and 'Email Addresses'
# Create empty DataFrames for clean and invalid data
Clean_SC_Data = pd.DataFrame(columns=SC_Data.columns)
Invalid_SC_Data = pd.DataFrame(columns=SC_Data.columns)

# Filter for rows with missing values in either column (using logical OR)
invalid_rows = SC_Data[(SC_Data['Contact Phone Number 1'].isna()) | (SC_Data['Email Addresses'].isna())]

# Keep rows with valid contact information
clean_rows = SC_Data.drop(invalid_rows.index)

# Fill the DataFrames
Clean_SC_Data = clean_rows
Invalid_SC_Data = invalid_rows

# Alternatively (using boolean indexing):
# Clean_SC_Data = SC_Data[SC_Data['Contact Phone Number 1'].notna() & SC_Data['Email Addresses'].notna()]
# Invalid_SC_Data = SC_Data[~Clean_SC_Data.index]  # Complement of Clean_SC_Data indices

print("Clean Data:")
print(Clean_SC_Data.head())  # Print the first few rows of Clean_SC_Data

print("Invalid Data:")
print(Invalid_SC_Data.head())  # Print the first few rows of Invalid_SC_Data

In [6]:
#-----EXPLORE-----
# Get column names as a list
column_names = PE_Data.columns.tolist()
#column_names = ASE_Data.columns.tolist()
#column_names = SC_Data.columns.tolist()

# Get data types as a Series (easier to work with)
data_types = PE_Data.dtypes
#data_types = ASE_Data.dtypes
#data_types = SC_Data.dtypes

# Print column names and data types together
for name, dtype in zip(column_names, data_types):
  print(f"Column Name: {name}, Data Type: {dtype}")

Column Name: *SFANameorLEAName, Data Type: object
Column Name: *AUN, Data Type: object
Column Name: *SchoolBuildingName, Data Type: object
Column Name: *School/Branch, Data Type: int64
Column Name: *PASecureID, Data Type: int64
Column Name: *StudentFirstName, Data Type: object
Column Name: StudentMiddleName, Data Type: object
Column Name: *StudentLastName, Data Type: object
Column Name: *StudentDateOfBirth, Data Type: object
Column Name: *MailingAddressLine1, Data Type: object
Column Name: MailingAddressLine2, Data Type: object
Column Name: Apt#, Data Type: object
Column Name: *City, Data Type: object
Column Name: *State, Data Type: object
Column Name: *ZipCode, Data Type: object
Column Name: *Eligibility, Data Type: object
Column Name: *AddressCountyName, Data Type: object
Column Name: Case Number (if available), Data Type: float64
Column Name: *PrimaryGuardianFirstName, Data Type: object
Column Name: *PrimaryGuardianLastName, Data Type: object
Column Name: *PrimaryGuardianPhoneNumber